In [0]:
!pip install transformers
!pip install torch

In [0]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import mlflow
import os

In [0]:
model_download_dir = "/Workspace/Users/tharushi.edinushika@gmail.com/mlflow-databricks/model"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

tokenizor.save_pretrained(model_download_dir)
model.save_pretrained(model_download_dir, max_shard_size="1GB")

In [0]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


In [0]:
print(response)

In [0]:
class QwenInstructModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        import torch

        model_dir = context.artifacts["model_dir"]
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_dir, torch_dtype=torch.bfloat16
        )
        self.model.eval()

        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token

        self.system_prompt = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."

    def predict(self, context, model_input: pd.DataFrame):
        import torch

        prompts = model_input["text"].tolist()
        responses = []

        for prompt in prompts:
            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt},
            ]
            text = self.tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            model_inputs = self.tok([text], return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                generated_ids = self.model.generate(
                    **model_inputs,
                    max_new_tokens=512,
                )

            generated_ids = [
                output_ids[len(input_ids):]
                for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            response = self.tok.batch_decode(generated_ids, skip_special_tokens=True)[0]
            responses.append(response)

        return pd.DataFrame({"predictions": responses})

In [0]:
os.getcwd()

In [0]:
mlflow.set_experiment('/Workspace/Users/tharushi.edinushika@gmail.com/mlflow-databricks/experiment/logging_qwen_model')

In [0]:
cpu_requirements = [
    "--index-url https://download.pytorch.org/whl/cpu",
    "--extra-index-url https://pypi.org/simple/",
    "torch",
    "transformers>=4.40.0",
    "pandas>=2.0",
    "numpy",
]

with mlflow.start_run() as run:
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=QwenInstructModel(),
        artifacts={"model_dir": model_download_dir},
        input_example=pd.DataFrame({"text": ["Give me a short introduction to large language models."]}),
        pip_requirements=cpu_requirements,
    )

In [0]:
print(f"Logged model URI: {model_info.model_uri}")
print(f"Run ID: {run.info.run_id}")

In [0]:
model_name = "workspace.default.qwen_instruct_model"
new_version = mlflow.register_model(model_info.model_uri, model_name)
print(f"Registered {model_name} version {new_version.version}")
print("Update the serving endpoint to this version and redeploy.")